In [2]:
import tkinter as tk
from tkinter import scrolledtext
import pyperclip
import re
import pandas as pd
import sys
from datetime import timedelta, datetime
from threading import Thread
import keyboard
import pygetwindow as gw
import time
import json
from PIL import Image, ImageTk

In [3]:
import getpass # для определения текущего пользователя позже убрать
env = getpass.getuser()

In [4]:
# 0.1 Куда сохранять эксель
xlsx_path = r"C:\Users\lutzb\Desktop\wt_stats\data.xlsx" if env == 'lutzb' else r"D:\py\wt_stats\data.xlsx"

In [5]:
with pd.ExcelFile(xlsx_path, engine='openpyxl') as xls:
    # Пытаемся прочитать лист 'battles'
    if 'battles' in xls.sheet_names:
        df_for_start_page = pd.read_excel(xls, sheet_name='vehicles', engine='openpyxl')
    else:
        print('ошибка')


In [6]:
df_for_start_page.tail(10)

,battle_id,result,vehicle,premium,sl_corrected,rp_corrected,mp,activity_percent,mission_time,kills,kills_air,crits,assists,base_caps,doubler_used,did_died
1209,5733630000ea127,Победа,Type 62,0,9352,3086,871,89.0,0.006181,1,0,3,2,1,0.0,1.0
1210,5733630000ea127,Победа,M41A3(Китай),1,720,60,0,0.0,0.000961,0,0,0,0,0,0.0,0.0
1211,5733a5f000eb9d9,Победа,M551,0,11866,2364,953,94.0,0.003380,2,0,4,2,0,0.0,0.0
1212,5733a5f000eb9d9,Победа,T58,1,41625,5715,1239,95.0,0.003426,5,0,5,0,0,0.0,1.0
1213,5733eee000ed369,Поражение,Т-72А,0,5825,618,497,62.0,0.000903,1,0,2,1,0,0.0,1.0
1214,5733eee000ed369,Поражение,БМП-3,0,6224,806,541,72.0,0.001250,1,0,2,1,0,0.0,1.0
1215,5733eee000ed369,Поражение,Т-62М-1,0,4941,504,320,53.0,0.000637,1,0,1,0,0,0.0,1.0
1216,5733eee000ed369,Поражение,ЗСУ-23-4М4,0,1700,274,0,41.0,0.001192,0,0,0,0,0,0.0,1.0
1217,5733eee000ed369,Поражение,ЗСУ-37-2,0,6485,894,280,76.0,0.001632,0,1,1,0,0,0.0,1.0
1218,5733eee000ed369,Поражение,2С25,0,3357,382,133,53.0,0.000648,0,0,0,1,0,0.0,1.0


In [7]:
# Добавляем столбец objectives
df_for_start_page['objectives'] = (
    df_for_start_page['kills'] +
    df_for_start_page['kills_air'] +
    df_for_start_page['crits'] +
    df_for_start_page['assists'] +
    df_for_start_page['base_caps']
)
# Добавляем столбец k/d
df_for_start_page['kd'] = (
    (df_for_start_page['kills'] + df_for_start_page['kills_air'] ) /
    (df_for_start_page['did_died'] + df_for_start_page['doubler_used'])
)

In [10]:
# Новый датафрейм через groupby по имени машинки
df_for_start_page = df_for_start_page.groupby('vehicle').agg(
    avg_sl = ('sl_corrected', 'mean'),
    avg_rp = ('rp_corrected', 'mean'),
    avg_mp = ('mp', 'mean'),
    battle_count = ('battle_id', 'count'),
    objectives = ('objectives', 'mean'),
    avg_kd = ('kd', 'mean'),
    total_did_died=('did_died', 'sum'),
    total_doubler_used=('doubler_used', 'sum')
)

# Добавить выживаемость
df_for_start_page['survivability'] = (
    (df_for_start_page['battle_count'] - df_for_start_page['total_did_died']) / df_for_start_page['battle_count']
)

df_for_start_page

,avg_sl,avg_rp,avg_mp,battle_count,objectives,avg_kd,total_did_died,total_doubler_used,survivability
vehicle,,,,,,,,,
2С25,4641.875000,827.812500,263.500000,16,1.687500,0.0,1.0,0.0,0.937500
2С3М,2168.625000,396.625000,146.500000,8,0.500000,NaN,0.0,0.0,1.000000
AD-4,3476.800000,558.600000,322.200000,5,2.400000,NaN,0.0,0.0,1.000000
AMC.35 (ACG.1),561.333333,520.333333,506.666667,3,3.000000,NaN,0.0,0.0,1.000000
AMD.35,305.142857,555.857143,354.285714,7,2.000000,NaN,0.0,0.0,1.000000
...,...,...,...,...,...,...,...,...,...
Т-62М-1,4917.833333,894.333333,319.166667,6,2.166667,1.0,1.0,0.0,0.833333
Т-72А,5439.466667,970.133333,322.600000,15,1.933333,1.0,1.0,0.0,0.933333
Як-9УТ,5510.600000,569.900000,432.200000,10,3.000000,NaN,0.0,0.0,1.000000


In [21]:
df_for_start_page = df_for_start_page[df_for_start_page['battle_count'] >= 5]

In [ ]:
for index, i in df_for_start_page.iterrows():
    if i['avg_sl'] == max(df_for_start_page['avg_sl']):
        print(index)

In [ ]:
df_for_start_page['avg_sl'].idxmax()

'T58'

In [63]:
label = ''

a = df_for_start_page.nlargest(3, 'avg_mp')
for index, row in a.iterrows():
    label += index
    label += round(row['avg_mp'])

label

TypeError: can only concatenate str (not "int") to str

In [13]:
# Найти индекс строки с максимальным значением
max_index = df_for_start_page['avg_sl'].idxmax()
max_value = df_for_start_page.loc[max_index, 'avg_sl']
print(f"Максимум: {max_value} в строке с индексом {max_index}")

Максимум: 28185.375 в строке с индексом T58


In [14]:
# Найти максимумы по всем числовым колонкам
numeric_cols = ['avg_sl', 'avg_rp', 'avg_mp']
max_values = df_for_start_page[numeric_cols].max()
print(max_values)

avg_sl    28185.375000
avg_rp     4838.156250
avg_mp     1229.315789
dtype: float64


In [41]:
# Найти индекс строки с максимальным значением
top_sl_name = df_for_start_page['avg_sl'].idxmax()
top_sl_value = round(df_for_start_page.loc[top_sl_name, 'avg_sl'])

top_rp_name = df_for_start_page['avg_rp'].idxmax()
top_rp_value = round(df_for_start_page.loc[top_rp_name, 'avg_rp'])

top_mp_name = df_for_start_page['avg_mp'].idxmax()
top_mp_value = round(df_for_start_page.loc[top_mp_name, 'avg_mp'])


print(f"""
      По SL - {top_sl_name} ({top_sl_value:_})
      По RP - {top_rp_name} ({top_rp_value:_})
      По MP - {top_mp_name} ({top_mp_value:_})
      """.replace("_", " "))


      По SL - T58 (28 185)
      По RP - T58 (4 838)
      По MP - M-51 (1 229)
      


In [17]:
df_for_start_page.query("""
    battle_count >= 5 & 
    avg_sl > 1000 &
    vehicle.str.contains('ИС')
""", engine='python')

ValueError: multi-line expressions are only valid in the context of data, use DataFrame.eval